## Librairies

In [1]:
import numpy as np
import cvxpy as cp
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from scipy.optimize import minimize
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.4f}'.format

## Functions

In [2]:
def compute_vt_price(price_ts, exp_window, target_vol, init_vt_price = 100):
    price_df = price_ts.to_frame(name='risky_asset')
    log_returns_df = np.log(price_df).diff()
    rolling_std_df = log_returns_df.ewm(span=exp_window, min_periods=exp_window, adjust=False).std()
    rolling_std_df *= (252 ** 0.5)
    leverage_df = target_vol / rolling_std_df
    vt_price_df = (price_df.loc[leverage_df.dropna().index].pct_change() * leverage_df.shift(1)).fillna(0).add(1).cumprod() * init_vt_price
    vt_price_df.columns = ['F']
    agg_df = pd.concat([price_df, leverage_df['risky_asset'].to_frame('beta'), vt_price_df], axis=1).dropna()
    agg_df['vt_delta'] = (agg_df['F'] * agg_df['beta'])/agg_df['risky_asset']
    return agg_df

def apply_compute_option(row):
    inputs = row[['F', 'K', 'T', 'r', 'sigma', 'option_type', 'compute_greeks', 'slide_scenario', 'slide_compute']].to_dict()
    return BSMModel.compute_option(**inputs)

def compute_vt_option_bt(agg_df, day_to_maturity, strike_delta, target_vol, slide_scenario):
    
    results = list()
    for date in tqdm(agg_df.index):
        agg_df_temp = agg_df.loc[date:].iloc[:day_to_maturity+1]
        if agg_df_temp.shape[0] != day_to_maturity+1: break
        agg_df_temp['strike_date'] = agg_df_temp.index[0]
        agg_df_temp['maturity_date'] = agg_df_temp.index[-1]
        agg_df_temp['days_to_maturity'] = list(range(agg_df_temp.shape[0]))[::-1]
        agg_df_temp['T'] = agg_df_temp['days_to_maturity'] / 252

        option_type = 'call' if strike_delta > 0 else 'put'
        agg_df_temp['option_type'] = option_type

        strike_k = BSMModel.solve_delta_strike(F=100, T=day_to_maturity/252, sigma=target_vol, r=0, option_type=option_type, target_delta=strike_delta)
        strike_pct = strike_k / 100
        agg_df_temp['K'] = agg_df_temp['F'].iloc[0] * strike_pct
        agg_df_temp['r'] = 0
        agg_df_temp['sigma'] = target_vol
        agg_df_temp['compute_greeks'] = True
        agg_df_temp['slide_scenario'] = slide_scenario
        agg_df_temp['slide_compute'] = 'option_pnl'
        
        pricing_df = agg_df_temp.apply(lambda x: apply_compute_option(x), axis=1, result_type='expand')
        agg_df_temp = pd.concat([agg_df_temp, pricing_df], axis=1)
        agg_df_temp['asset_delta'] = agg_df_temp['vt_delta'] * agg_df_temp['delta']
        agg_df_temp['asset_delta_cash'] = agg_df_temp['risky_asset'] * agg_df_temp['asset_delta']
        agg_df_temp['dP'] = agg_df_temp['price'].diff()
        agg_df_temp['dH'] = agg_df_temp['risky_asset'].diff() * agg_df_temp['asset_delta'].shift(1)
        results.append(agg_df_temp)
    results_df = pd.concat(results)
    return results_df

def compute_scaling(trading_units, trading_scale, results_df, slide_scenario, day_to_maturity, cols=['asset_delta_cash','dP', 'dH']):
    scaling_factor = trading_units / results_df.groupby('strike_date')[trading_scale].first()
    results_df['scaling_factor'] = results_df['strike_date'].map(scaling_factor)
    for col in cols:
        results_df[f'scaled_{col}'] = results_df['scaling_factor'] * results_df[col]
    results_df[f'scaled_{trading_scale}'] = results_df['scaling_factor'] * results_df[trading_scale]
    results_df[f'scaled_{slide_scenario}'] = results_df['scaling_factor'] * results_df[slide_scenario]
    nb_strike = results_df.groupby(results_df.index)['strike_date'].count()
    mask = (nb_strike == (day_to_maturity+1))
    results_df = results_df.loc[mask]
    return results_df

def optimize_portfolio(returns, solver_type="SLSQP", target_return=None, risk_free_rate=0.0):
    """
    Optimizes portfolio weights to maximize the Sharpe Ratio (or minimize variance for QP).
    
    Args:
        prices (pd.DataFrame): Time series of asset prices (columns=assets, index=date).
        solver_type (str): 'SLSQP', 'CVXPY', 'ANALYTICAL', 'QP', 'MONTECARLO'.
        target_return (float): Required if solver_type='QP'. Annualized target return.
        risk_free_rate (float): Annualized risk-free rate (default 0.0).
        
    Returns:
        np.array: Optimal weights for the assets.
    """
    # --- 1. Data Prep ---
    # Calculate daily returns
    #returns = prices.pct_change().dropna()
    
    # Annualized mean returns and covariance (Assuming 252 trading days)
    mu = returns.mean() * 252
    sigma = returns.cov() * 252
    n_assets = len(mu)
    
    # Helper to calculate portfolio stats for a given weight vector
    def get_stats(w):
        w = np.array(w)
        ret = np.sum(mu * w)
        vol = np.sqrt(np.dot(w.T, np.dot(sigma, w)))
        sr = (ret - risk_free_rate) / vol if vol > 0 else 0
        return ret, vol, sr

    # --- 2. Solver Implementations ---

    if solver_type == "ANALYTICAL":
        # Mathematical Solution (Unconstrained - Allows Short Selling)
        # Formula: z = Sigma^-1 * (mu - rf); w = z / sum(z)
        inv_sigma = np.linalg.inv(sigma)
        excess_mu = mu - risk_free_rate
        
        # Calculate unscaled weights (z)
        z = np.dot(inv_sigma, excess_mu)
        # Normalize so sum(weights) = 1
        w_opt = z / np.sum(z)
        return w_opt

    elif solver_type == "SLSQP":
        # Numerical Optimization (Constrained: Long-only)
        
        # Objective: Minimize Negative Sharpe Ratio
        def neg_sharpe(w):
            return -get_stats(w)[2]
        
        # Constraints: Sum of weights = 1
        constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        # Bounds: 0 <= w <= 1 (No short selling)
        bounds = tuple((0, 1) for _ in range(n_assets))
        init_guess = n_assets * [1. / n_assets]
        
        result = minimize(neg_sharpe, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return result.x

    elif solver_type == "CVXPY":
        # Convex Optimization (Robust Long-only)
        # Transformation: Minimize (1/2)x'Ex subject to (mu-rf)'x = 1, x >= 0. 
        # Then w = x / sum(x).
        
        x = cp.Variable(n_assets)
        
        # Objective: Minimize unnormalized variance
        objective = cp.Minimize(cp.quad_form(x, sigma))
        
        # Constraints
        constraints = [
            (mu.values - risk_free_rate) @ x == 1,  # Set excess return scale to 1
            x >= 0                                  # Long only
        ]
        
        prob = cp.Problem(objective, constraints)
        prob.solve()
        
        # Recover actual weights
        w_opt = x.value / np.sum(x.value)
        return w_opt

    elif solver_type == "QP":
        # Quadratic Programming (Target Return)
        # Minimizes Variance subject to a specific Target Return
        if target_return is None:
            raise ValueError("For solver_type='QP', you must provide a 'target_return'.")
            
        def portfolio_variance(w):
            return np.dot(w.T, np.dot(sigma, w))
            
        constraints = (
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},              # Sum weights = 1
            {'type': 'eq', 'fun': lambda w: np.sum(w * mu) - target_return} # Portfolio Return = Target
        )
        bounds = tuple((0, 1) for _ in range(n_assets))
        init_guess = n_assets * [1. / n_assets]
        
        result = minimize(portfolio_variance, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return result.x

    elif solver_type == "MONTECARLO":
        # Brute Force Simulation
        num_portfolios = 20000
        best_sr = -np.inf
        best_w = None
        
        for _ in range(num_portfolios):
            w = np.random.random(n_assets)
            w /= np.sum(w) # Normalize
            _, _, sr = get_stats(w)
            
            if sr > best_sr:
                best_sr = sr
                best_w = w
                
        return best_w

    else:
        raise ValueError(f"Unknown solver type: {solver_type}")
        

## Inputs

In [3]:
pairs = [['SPY', 'QQQ'], ['EWQ', 'EWG']]

exp_window = 20
target_vol = 0.10
strike_delta = -0.1
day_to_maturity = 250
slide_scenario = -0.3

# scaling
trading_units = 20 * (10 / day_to_maturity)
trading_scale = 'theta' # 'F', 'price', 'delta', 'gamma', 'vega', 'theta', 'vanna', 'volga', slide_scenario

## Script

In [4]:
# load daily returns
gross_daily_pnl = dict()
for pair in pairs:
    for symbol in pair:
        print(f'loading symbol {symbol}')
        price_ts = pd.read_csv(f'data/{symbol}.csv', index_col=0, parse_dates=True)['price']
        agg_df = compute_vt_price(price_ts, exp_window, target_vol)
        results_df = compute_vt_option_bt(agg_df, day_to_maturity, strike_delta, target_vol, slide_scenario)
        scaled_res_df = compute_scaling(trading_units, trading_scale, results_df, slide_scenario, day_to_maturity)
        gross_daily_pnl[symbol] = scaled_res_df.groupby(scaled_res_df.index)['scaled_dH'].sum()
gross_daily_pnl_df = pd.DataFrame(gross_daily_pnl)

loading symbol SPY


 96%|█████████▋| 6499/6749 [13:10<00:30,  8.22it/s]


loading symbol QQQ


 96%|█████████▋| 6454/6704 [6:58:23<16:12,  3.89s/it]     


loading symbol EWQ


 95%|█████████▍| 4425/4675 [08:53<00:30,  8.29it/s]


loading symbol EWG


 95%|█████████▍| 4425/4675 [08:53<00:30,  8.30it/s]


In [5]:
gross_daily_pnl_df.describe()

,SPY,QQQ,EWQ,EWG
count,"6,249.0000","6,204.0000","4,175.0000","4,175.0000"
mean,95.8456,100.0994,101.9579,35.8153
std,"4,348.1867","3,979.3492","5,220.1635","6,392.8507"
min,"-51,227.8322","-44,620.5479","-47,461.1624","-52,714.2744"
25%,-615.7854,-605.8266,-902.5834,-920.3109
50%,81.5908,81.1039,85.8261,81.4831
75%,838.9556,763.3581,"1,079.1152","1,054.3256"
max,"77,101.0273","70,544.9664","70,255.8244","91,929.2529"


In [9]:
px.line(gross_daily_pnl_df.cumsum())

In [11]:
for pair in pairs:

    print(f"pair: {pair}")
    returns = gross_daily_pnl_df.loc[:, pair]

    mu = returns.mean() * 0
    mu = mu+1
    
    sigma = returns.cov() * 252
    n_assets = len(mu)

    inv_sigma = np.linalg.inv(sigma)

    # Calculate unscaled weights (z)
    z = np.dot(inv_sigma, mu)

    # Normalize so sum(weights) = 1
    w_opt = z / np.sum(z)
    print(f"weights: {w_opt}")
    display(px.line(returns.dot(w_opt).dropna().cumsum()))
    strategy_returns = returns.dot(w_opt).dropna()
    print('display returns')
    display(strategy_returns.groupby(strategy_returns.index.year).sum())

pair: ['SPY', 'QQQ']
weights: [0.22100401 0.77899599]


display returns


2000    -7,462.3926
2001   -38,882.4792
2002    -8,550.7654
2003    50,589.6799
2004    30,379.8308
2005    25,514.5229
2006    34,316.1058
2007    28,648.2917
2008   -50,384.9079
2009    79,613.5878
2010    29,853.8613
2011    25,719.7276
2012    31,874.1484
2013    38,251.1654
2014    25,726.5541
2015    30,862.9054
2016    38,410.9532
2017    36,854.2302
2018     5,661.4617
2019    61,859.8683
2020    28,949.8022
2021    29,079.6042
2022    15,309.1286
2023    47,439.4913
2024    24,558.2120
dtype: float64

pair: ['EWQ', 'EWG']
weights: [ 1.23982525 -0.23982525]


display returns


2008   -82,753.5085
2009    12,175.9748
2010    37,818.5316
2011    22,534.7659
2012    49,540.2084
2013    28,023.6956
2014   -57,260.8512
2015   102,601.2775
2016    65,724.8945
2017    29,484.1825
2018   -57,895.4734
2019   133,524.4257
2020    42,688.5908
2021    32,734.5931
2022    86,723.3164
2023    32,168.1077
2024    14,068.0224
dtype: float64